# **CatBoost**

тру ля ля 

## Предобработка

In [1]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [2]:
import pandas as pd
from data_preprocessing.DataForModel import build_target, split_data, preprocess_dataset

In [ ]:
name = input("Введите имя файла большими буквами: ")
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
df = preprocess_dataset(df)

In [ ]:
df = build_target(df, h=20, target_type="classification")

y = df["GoodTrade"]

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, target="GoodTrade", val_size=0.1, test_size=0.2, split_type="train_test")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Обучение

In [ ]:
model_cb = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=5,
    learning_rate=0.05,
    iterations=500,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=50,          # лог каждые 50 итераций
    use_best_model=True  # запомнит лучшую итерацию по AUC на валидации
)

model_cb.fit(
    X_train, y_train,
    cat_features=cat_cols,        # имена категориальных столбцов
    eval_set=(X_test, y_test)
)


0:	test: 0.5281519	best: 0.5281519 (0)	total: 54.6ms	remaining: 27.2s
50:	test: 0.5276988	best: 0.5301515 (33)	total: 293ms	remaining: 2.58s
100:	test: 0.5414466	best: 0.5419153 (99)	total: 525ms	remaining: 2.08s
150:	test: 0.5463209	best: 0.5468833 (142)	total: 777ms	remaining: 1.79s
200:	test: 0.5517263	best: 0.5552570 (180)	total: 1.01s	remaining: 1.5s
250:	test: 0.5419466	best: 0.5552570 (180)	total: 1.24s	remaining: 1.23s
300:	test: 0.5497266	best: 0.5552570 (180)	total: 1.47s	remaining: 973ms
350:	test: 0.5445712	best: 0.5552570 (180)	total: 1.7s	remaining: 720ms
400:	test: 0.5446649	best: 0.5552570 (180)	total: 1.96s	remaining: 483ms
450:	test: 0.5463209	best: 0.5552570 (180)	total: 2.2s	remaining: 239ms
499:	test: 0.5484768	best: 0.5552570 (180)	total: 2.43s	remaining: 0us

bestTest = 0.5552569911
bestIteration = 180

Shrink model to first 181 iterations.


In [ ]:
proba_cb = model_cb.predict_proba(X_test)[:, 1]
y_pred_cb = (proba_cb >= 0.5).astype(int)

print("CatBoost AUC:", roc_auc_score(y_test, proba_cb))
print("\nОтчёт по классификации (CatBoost):")
print(classification_report(y_test, y_pred_cb))
print("Матрица ошибок (CatBoost):")
print(confusion_matrix(y_test, y_pred_cb))


CatBoost AUC: 0.5552569910951414

Отчёт по классификации (CatBoost):
              precision    recall  f1-score   support

           0       0.54      0.66      0.60       185
           1       0.53      0.40      0.46       173

    accuracy                           0.54       358
   macro avg       0.54      0.53      0.53       358
weighted avg       0.54      0.54      0.53       358

Матрица ошибок (CatBoost):
[[123  62]
 [103  70]]
